# Practice Lab: Back Propagation Using a Computation Graph

This notebook works through the same core idea as the original backprop lab — but with a brand new example: **predicting a taxi fare**. 

Gradient descent needs the derivative of the cost with respect to every parameter in a network. For a network with millions of parameters, we compute those derivatives efficiently using **backpropagation**, and we organize the computation using a **computation graph** — breaking one big expression into small, simple steps, computing derivatives locally at each step, then chaining them together (the **chain rule**).

We'll do this two ways for every derivative:
1. **Arithmetically** — nudge a variable by a tiny amount ($\epsilon$) and see how the output changes.
2. **Symbolically** — use SymPy to compute the exact derivative.

Both should agree (up to the small approximation error from using a finite $\epsilon$ instead of an infinitesimally small one).

In [1]:
from sympy import symbols, diff
import numpy as np

## Part 1: Warm-Up — A Simple Computation Graph

Let's start with a slightly complex expression: $J = (5 + 2w)^2$. We want $\frac{\partial J}{\partial w}$.

We break it into two nodes:

```
w ──(a = 5 + 2w)──▶ a ──(J = a²)──▶ J
```

### Forward propagation
First, compute the values moving left to right.

In [2]:
w = 4
a = 5 + 2*w
J = a**2
print(f"a = {a}, J = {J}")

a = 13, J = 169


### Backpropagation

Backprop moves right to left. The first node we hit is $J = a^2$, so the first derivative we need is $\frac{\partial J}{\partial a}$.

#### $\frac{\partial J}{\partial a}$ — Arithmetically
Nudge $a$ by a tiny $\epsilon$ and see how $J$ changes.

In [3]:
a_epsilon = a + 0.001
J_epsilon = a_epsilon**2
k = (J_epsilon - J) / 0.001
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_da ~= k = {k}")

J = 169, J_epsilon = 169.02600099999998, dJ_da ~= k = 26.000999999979513


#### $\frac{\partial J}{\partial a}$ — Symbolically

We prefix symbolic variables with `s` (e.g. `sa` for symbolic $a$).

In [4]:
sw, sJ, sa = symbols('w,J,a')
sJ = sa**2
sJ

a**2

In [5]:
sJ.subs([(sa, a)])

169

In [6]:
dJ_da = diff(sJ, sa)
dJ_da

2*a

So $\frac{\partial J}{\partial a} = 2a$. With $a = 13$, that's $26$ — matching the arithmetic result above.

#### $\frac{\partial a}{\partial w}$ — Arithmetically

Next, moving further left, we need $\frac{\partial a}{\partial w}$.

In [7]:
w_epsilon = w + 0.001
a_epsilon = 5 + 2*w_epsilon
k = (a_epsilon - a) / 0.001
print(f"a = {a}, a_epsilon = {a_epsilon}, da_dw ~= k = {k}")

a = 13, a_epsilon = 13.002, da_dw ~= k = 2.000000000000668


#### $\frac{\partial a}{\partial w}$ — Symbolically

In [8]:
sa = 5 + 2*sw
sa

2*w + 5

In [9]:
da_dw = diff(sa, sw)
da_dw

2

### The Chain Rule

A small change in $w$ changes $a$ by $2\times$ that amount. A small change in $a$ changes $J$ by $2a\times$ that amount. So a small change in $w$ changes $J$ by $2 \times 2a$ times that amount:

$$\frac{\partial J}{\partial w} = \frac{\partial a}{\partial w} \cdot \frac{\partial J}{\partial a}$$

In [10]:
dJ_dw = da_dw * dJ_da
dJ_dw

4*a

With $a = 13$, $\frac{\partial J}{\partial w} = 2 \times 26 = 52$. Let's verify arithmetically.

In [11]:
w_epsilon = w + 0.001
a_epsilon = 5 + 2*w_epsilon
J_epsilon = a_epsilon**2
k = (J_epsilon - J) / 0.001
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_dw ~= k = {k}")

J = 169, J_epsilon = 169.052004, dJ_dw ~= k = 52.00400000001082


They match. Now let's apply the same process to something more realistic.

## Part 2: Computation Graph of a Simple Neuron — Predicting a Taxi Fare

Imagine a tiny "neuron" that predicts a taxi fare from the distance travelled:

- $x$ = distance travelled, in miles (input)
- $w$ = price per mile (parameter)
- $b$ = base/flag-down fare (parameter)
- $y$ = the actual fare that was paid (target/label)
- $a$ = the model's predicted fare
- $d$ = prediction error
- $J$ = the cost (we use $\frac{1}{2}d^2$, the $\frac{1}{2}$ makes the derivative clean)

As a computation graph:

```
x ──┐
    ├──(c = w·x)──▶ c ──┐
w ──┘                   ├──(a = c + b)──▶ a ──┐
                    b ──┘                     ├──(d = a − y)──▶ d ──(J = d²/2)──▶ J
                                          y ──┘
```

Our goal, just like in real training, is $\frac{\partial J}{\partial w}$ and $\frac{\partial J}{\partial b}$ — how the cost changes with respect to each *parameter*. We don't need $\frac{\partial J}{\partial x}$ or $\frac{\partial J}{\partial y}$ since those aren't learnable.

### Forward propagation

In [12]:
# Inputs and parameters
x = 4    # distance travelled, in miles
w = 2    # price per mile
b = 3    # base fare
y = 7    # actual fare paid

# forward pass, step by step
c = w * x
a = c + b
d = a - y
J = d**2 / 2
print(f"J = {J}, d = {d}, a = {a}, c = {c}")

J = 8.0, d = 4, a = 11, c = 8


The model predicted a fare of \$11 when the actual fare was \$7 — so the cost captures that \$4 overestimate.

### Backward propagation

We work right to left. At each node we find the **local derivative**, then combine it with the derivative already computed to its right using the chain rule.

#### $\frac{\partial J}{\partial d}$ — Arithmetically

In [13]:
d_epsilon = d + 0.001
J_epsilon = d_epsilon**2 / 2
k = (J_epsilon - J) / 0.001
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_dd ~= k = {k}")

J = 8.0, J_epsilon = 8.004000500000002, dJ_dd ~= k = 4.00050000000185


#### $\frac{\partial J}{\partial d}$ — Symbolically

In [14]:
sx, sw, sb, sy, sJ = symbols('x,w,b,y,J')
sa, sc, sd = symbols('a,c,d')
sJ = sd**2 / 2
sJ

d**2/2

In [15]:
sJ.subs([(sd, d)])

8

In [16]:
dJ_dd = diff(sJ, sd)
dJ_dd

d

So $\frac{\partial J}{\partial d} = d = 4$, matching the arithmetic result.

#### $\frac{\partial J}{\partial a}$

We first need the local derivative $\frac{\partial d}{\partial a}$ (note: $d = a - y$, and we don't care about $\frac{\partial d}{\partial y}$ since $y$ isn't a parameter).

##### Arithmetically

In [17]:
a_epsilon = a + 0.001
d_epsilon = a_epsilon - y
k = (d_epsilon - d) / 0.001
print(f"d = {d}, d_epsilon = {d_epsilon}, dd_da ~= k = {k}")

d = 4, d_epsilon = 4.0009999999999994, dd_da ~= k = 0.9999999999994458


##### Symbolically

In [18]:
sd = sa - sy
sd

a - y

In [19]:
dd_da = diff(sd, sa)
dd_da

1

Now chain it together: $\frac{\partial J}{\partial a} = \frac{\partial d}{\partial a} \cdot \frac{\partial J}{\partial d}$

In [20]:
dJ_da = dd_da * dJ_dd
dJ_da

d

Let's verify this arithmetically too.

In [21]:
a_epsilon = a + 0.001
d_epsilon = a_epsilon - y
J_epsilon = d_epsilon**2 / 2
k = (J_epsilon - J) / 0.001
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_da ~= k = {k}")

J = 8.0, J_epsilon = 8.004000499999998, dJ_da ~= k = 4.000499999998297


They match. From here on we'll move a bit faster since the pattern is familiar.

#### $\frac{\partial J}{\partial c}$ and $\frac{\partial J}{\partial b}$

The node $a = c + b$ has two local derivatives of interest: $\frac{\partial a}{\partial c}$ (so we can keep propagating left) and $\frac{\partial a}{\partial b}$ (since $b$ is a parameter we actually care about).

In [22]:
sa = sc + sb
sa

b + c

In [23]:
da_dc = diff(sa, sc)
da_db = diff(sa, sb)
print(da_dc, da_db)

1 1


In [24]:
dJ_dc = da_dc * dJ_da
dJ_db = da_db * dJ_da
print(f"dJ_dc = {dJ_dc}, dJ_db = {dJ_db}")

dJ_dc = d, dJ_db = d


So $\frac{\partial J}{\partial b} = 4$ — that's one of the two gradients gradient descent needs, done!

#### $\frac{\partial J}{\partial w}$

The last node computes $c = w \cdot x$. We are interested in how $J$ changes with respect to $w$ — we won't bother with $\frac{\partial J}{\partial x}$ since $x$ is just an input, not a parameter.

In [25]:
sc = sw * sx
sc

w*x

In [26]:
dc_dw = diff(sc, sw)
dc_dw

x

This local derivative equals $x$ — it'll vary depending on how far the ride was. Combine it with $\frac{\partial J}{\partial c}$:

In [27]:
dJ_dw = dc_dw * dJ_dc
dJ_dw

d*x

In [28]:
print(f"dJ_dw = {dJ_dw.subs([(sx, x)])}")

dJ_dw = 4*d


With $x = 4$, $\frac{\partial J}{\partial w} = 16$. Let's check this arithmetically:

In [29]:
J_epsilon = ((w + 0.001)*x + b - y)**2 / 2
k = (J_epsilon - J) / 0.001
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_dw ~= k = {k}")

J = 8.0, J_epsilon = 8.016007999999998, dJ_dw ~= k = 16.00799999999758


They match! With $\frac{\partial J}{\partial w} = 16$ and $\frac{\partial J}{\partial b} = 4$, gradient descent could now update both parameters:

$$w := w - \alpha \frac{\partial J}{\partial w}, \qquad b := b - \alpha \frac{\partial J}{\partial b}$$

to gradually shrink the prediction error.

## Summary

The steps of backprop, walking right to left through the graph, at every node:
1. Calculate the **local derivative(s)** of that node (the derivative of its output with respect to its own inputs).
2. Use the **chain rule** to combine the local derivative with the derivative already found for the node to its right.

Try changing `x`, `w`, `b`, and `y` in Part 2 above and re-running the notebook top to bottom — the arithmetic and symbolic derivatives should always agree, no matter what values you pick. You could also extend this graph by adding another layer (e.g. a second weight/feature) to see how the chain rule scales to deeper networks.

## Congratulations!
You've worked through backpropagation using a computation graph on a fresh example. The same node-by-node approach scales up to networks with millions of parameters — that's exactly what frameworks like TensorFlow and PyTorch automate for you.